# KSL Ablation Study — Table 4 (co-training)

Runs the five configurations of the paper's ablation table and records the
KSL head's val accuracy for each, matching the **co-training** protocol
(Section 3.7) -- your reported full-model number (93.58%) -- not the
sequential pretrain-then-finetune pipeline.

Co-training means a `ksl` head joins the multilingual round-robin directly
(alongside `arabic`, `argentina`, `rsl`), trained jointly from scratch in
ONE stage -- there is no separate Stage-1 checkpoint to warm-start from.
That actually makes this the fully clean ablation (no earlier discussion's
"only ablated at finetune time" shortcut applies): every config below
retrains the entire architecture from scratch with exactly one design
element removed.

| Row | What's disabled | How |
|---|---|---|
| Full model | nothing | baseline -- should reproduce ≈93.58% on the `ksl` head |
| − node-level visibility masking | `mask_aware=False` | absent joints, already zero-filled, are treated as ordinary nodes |
| − frame-usability rule | `frame_rule="strict"` | any frame missing a hand is masked, not just totally-empty frames |
| − anatomical adjacency | `fully_connected=True` | bones/fingers graph replaced by a fully-connected graph |
| − normalization-aware augmentation | `aug="generic"` | `anchor_jitter` / `reach_scale` dropped from the pipeline |

**Cost note:** each config is a full 500-epoch, 4-language joint training run
(same budget as `train.ipynb`'s `multilingual_pretrain` stage) -- five of
these back-to-back is a large commitment on a laptop. Use `EPOCHS_OVERRIDE`
below to smoke-test all 5 configs on a handful of epochs first.


## 1. Imports

In [ ]:
import sys, json, time
from pathlib import Path

_here = (Path("__file__").resolve().parent
         if "__file__" in dir() else Path(".").resolve())
_root = _here.parent
sys.path.insert(0, str(_root / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pandas as pd
from tqdm import tqdm

from ksl_recognition.data.augmentations import AugmentPipeline
from ksl_recognition.data.dataset import (KSLLandmarkDataset, PadCollate,
                                           scan_landmarks, scan_val_landmarks,
                                           stratified_split)
from ksl_recognition.models.sign_model import MultiHeadSignModel, count_parameters

print("torch", torch.__version__)


## 2. Configuration

Hyperparameters mirror `train.ipynb`'s `multilingual_pretrain` stage (the
co-training run uses the SAME pretrain-style schedule, just with a `ksl`
head added to the round-robin) -- keep these in sync with whatever produced
your reported 93.58%.

In [ ]:
LANGUAGES = {
    "rsl":       dict(data="../data/processed_landmarks/slovo/train", val_data="../data/processed_landmarks/slovo/test"),
    "arabic":    dict(data="../data/processed_landmarks/arabic",      val_data=None),
    "argentina": dict(data="../data/processed_landmarks/argentina",   val_data=None),
    "ksl":       dict(data="../data/processed_landmarks/ksl",         val_data=None),   # co-training head
}
STEPS_PER_LANG = 60

VAL_FRAC   = 0.15
SEED       = 67
BATCH_SIZE = 32
MAX_LEN    = 128
WORKERS    = 2   # x len(LANGUAGES) active processes -- same note as train.ipynb

# ── Optimiser hyperparameters -- pretrain-style schedule, from train.ipynb ─
EPOCHS         = 500
ENC_LR         = 5e-4
HEAD_LR        = 1e-3
WEIGHT_DECAY   = 4e-4
WARMUP_EPOCHS  = 10
SCHED_PATIENCE = 12
LABEL_SMOOTHING = 0.1

# ── Model architecture -- shared across stages, from train.ipynb section 2 ──
IN_CH      = 5           # x, y, dx, dy, vis
D_MODEL    = 256
GCN_HIDDEN = (128, 256)
NHEAD      = 8
NUM_LAYERS = 6
DIM_FF     = 512
DROPOUT    = 0.25

# Quick smoke-test: set to a small number (e.g. 3) to verify all 5 configs
# run end-to-end before committing to the full EPOCHS x 5 training budget.
EPOCHS_OVERRIDE = None   # e.g. 3 for a smoke test; None = use EPOCHS above

RESULTS_PATH = Path("../plots/ablation_results.json")
CKPT_DIR     = Path("../checkpoints/ablation")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
Path("../plots").mkdir(parents=True, exist_ok=True)

def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = pick_device()
print("device:", device)
print("epochs per config:", EPOCHS_OVERRIDE or EPOCHS)
print("languages:", list(LANGUAGES))


## 3. The five configurations

Each dict fully specifies one row of Table 4. `name` is used for checkpoint
filenames and the results table; `label` is the exact row wording from the
paper.

In [ ]:
ABLATIONS = [
    dict(name="full_model", label="Full model (mask-aware GCN + Transformer + multilingual pretrain)",
         mask_aware=True,  fully_connected=False, frame_rule="lenient", aug="norm_aware"),
    dict(name="no_visibility_masking", label="− node-level visibility masking (zero-fill absent joints instead)",
         mask_aware=False, fully_connected=False, frame_rule="lenient", aug="norm_aware"),
    dict(name="no_frame_usability_rule", label="− frame-usability rule (mask any frame missing a hand)",
         mask_aware=True,  fully_connected=False, frame_rule="strict",  aug="norm_aware"),
    dict(name="fully_connected_adjacency", label="− anatomical adjacency (fully-connected graph instead)",
         mask_aware=True,  fully_connected=True,  frame_rule="lenient", aug="norm_aware"),
    dict(name="no_normalization_aware_augmentation", label="− normalization-aware augmentation (default GCN augmentation only)",
         mask_aware=True,  fully_connected=False, frame_rule="lenient", aug="generic"),
]

for cfg in ABLATIONS:
    print(f"  {cfg['name']:38s} mask_aware={cfg['mask_aware']!s:5} "
          f"fully_connected={cfg['fully_connected']!s:5} "
          f"frame_rule={cfg['frame_rule']:8} aug={cfg['aug']}")


## 4. Data (scanned once per language -- shared across every config)

Mirrors `train.ipynb`'s `multilingual_pretrain` data-loading cell exactly.

In [ ]:
def _load_language(name, cfg):
    data_dir = Path(cfg["data"])
    if not data_dir.exists():
        print(f"  [{name}] SKIPPED -- {data_dir} not found (not extracted yet?)")
        return None
    items, class_to_idx = scan_landmarks(data_dir)
    if cfg.get("val_data"):
        train_items = items
        val_items = scan_val_landmarks(cfg["val_data"], class_to_idx)
    else:
        train_items, val_items = stratified_split(items, VAL_FRAC, SEED)
    print(f"  [{name:10s}] {len(class_to_idx):4d} classes | "
          f"{len(train_items):6d} train / {len(val_items):5d} val")
    return dict(class_to_idx=class_to_idx, num_classes=len(class_to_idx),
                train_items=train_items, val_items=val_items)

print(f"Scanning {len(LANGUAGES)} registered language(s):")
LANG_DATA = {}
for name, cfg in sorted(LANGUAGES.items()):
    result = _load_language(name, cfg)
    if result is not None:
        LANG_DATA[name] = result
if "ksl" not in LANG_DATA:
    raise RuntimeError("KSL data not found -- co-training needs the ksl head to actually train on KSL.")
print(f"\n{len(LANG_DATA)} active language(s): {sorted(LANG_DATA)}")


## 5. Per-config builders

Only the pieces each ablation actually touches get rebuilt per config:
augmenter (`aug`), collate's frame rule (`frame_rule`), and the model's
`mask_aware` / `fully_connected` switches -- applied identically across every
language's loader, since these are architecture/pipeline choices, not
KSL-specific ones. Everything else (data split, optimizer schedule,
architecture sizes) stays fixed across configs so the comparison isolates
just that one design element.

In [ ]:
def build_augmenter(cfg):
    base = AugmentPipeline.strong()
    if cfg["aug"] == "generic":
        return AugmentPipeline.without_normalization_aware(base)
    return base


def build_langs(cfg):
    augmenter = build_augmenter(cfg)
    collate = PadCollate(use_velocity=True, frame_rule=cfg["frame_rule"])
    langs = {}
    for name, data in LANG_DATA.items():
        train_ds = KSLLandmarkDataset(data["train_items"], data["class_to_idx"], MAX_LEN, augment=augmenter)
        val_ds   = KSLLandmarkDataset(data["val_items"],   data["class_to_idx"], MAX_LEN)
        langs[name] = dict(
            class_to_idx=data["class_to_idx"], num_classes=data["num_classes"],
            train_loader=DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                     collate_fn=collate, num_workers=WORKERS, drop_last=True),
            val_loader=DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                                   collate_fn=collate, num_workers=WORKERS, drop_last=True),
        )
    return langs


def build_model(cfg, langs):
    head_classes = {name: l["num_classes"] for name, l in langs.items()}
    model = MultiHeadSignModel(
        in_ch=IN_CH, head_classes=head_classes,
        d_model=D_MODEL, gcn_hidden=GCN_HIDDEN, nhead=NHEAD,
        num_layers=NUM_LAYERS, dim_ff=DIM_FF, dropout=DROPOUT, max_len=MAX_LEN,
        mask_aware=cfg["mask_aware"], fully_connected=cfg["fully_connected"],
    ).to(device)

    for m in model.modules():
        if isinstance(m, nn.TransformerEncoder):
            m.enable_nested_tensor = False
            m.use_nested_tensor = False
    return model


def build_optimizer(model):
    """Layer-wise LR: smaller on the shared encoder, larger on every head."""
    head_ids    = {id(p) for head in model.heads.values() for p in head.parameters()}
    head_params = [p for p in model.parameters() if id(p) in head_ids]
    enc_params  = [p for p in model.parameters() if id(p) not in head_ids]
    return torch.optim.AdamW(
        [{"params": enc_params, "lr": ENC_LR}, {"params": head_params, "lr": HEAD_LR}],
        weight_decay=WEIGHT_DECAY,
    )


def cycle_loader(loader):
    while True:
        for batch in loader:
            yield batch


@torch.no_grad()
def evaluate_all(model, langs):
    model.eval()
    results = {}
    for name, lang in langs.items():
        correct = total = 0
        for feats, vis, fmask, labels in lang["val_loader"]:
            feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
            logits = model(feats, vis, fmask, head=name)
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.numel()
        results[name] = correct / max(total, 1)
    return results


## 6. Training loop for one config

Joint round-robin training over every active language, mirroring
`train.ipynb`'s `multilingual_pretrain` branch. The checkpoint-selection
criterion is the SAME one the original run used: best **average** val
accuracy across all languages (not best KSL accuracy alone) -- so the number
reported for Table 4 is the `ksl` head's accuracy at that same epoch,
matching how your existing Table 3 numbers were derived.

In [ ]:
def run_ablation(cfg, epochs):
    print(f"\n=== {cfg['name']} ===")
    torch.manual_seed(SEED)
    langs = build_langs(cfg)
    model = build_model(cfg, langs)
    print(f"  trainable parameters: {count_parameters(model):,}")

    optim = build_optimizer(model)
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optim, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
    plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optim, mode="max", factor=0.5, patience=SCHED_PATIENCE, min_lr=1e-6)

    lang_names = sorted(langs)
    train_iters = {name: cycle_loader(l["train_loader"]) for name, l in langs.items()}
    steps_per_epoch = STEPS_PER_LANG * len(lang_names)

    best_avg_acc = 0.0
    best_ksl_acc = 0.0
    ckpt_path = CKPT_DIR / f"{cfg['name']}.pt"
    epoch_bar = tqdm(range(1, epochs + 1), desc=cfg["name"], unit="epoch")
    for epoch in epoch_bar:
        model.train()
        for step in range(steps_per_epoch):
            name = lang_names[step % len(lang_names)]
            feats, vis, fmask, labels = next(train_iters[name])
            feats, vis, fmask, labels = (t.to(device) for t in (feats, vis, fmask, labels))
            logits = model(feats, vis, fmask, head=name)
            loss = F.cross_entropy(logits, labels, label_smoothing=LABEL_SMOOTHING)
            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        val_results = evaluate_all(model, langs)
        avg_val_acc = sum(val_results.values()) / len(val_results)
        if epoch <= WARMUP_EPOCHS:
            warmup_sched.step()
        else:
            plateau_sched.step(avg_val_acc)

        if avg_val_acc > best_avg_acc:
            best_avg_acc = avg_val_acc
            best_ksl_acc = val_results['ksl']
            torch.save({"model": model.state_dict(),
                        "class_to_idx": {n: l["class_to_idx"] for n, l in langs.items()},
                        "avg_val_acc": avg_val_acc, "per_language_val": val_results,
                        "epoch": epoch, "config": cfg}, ckpt_path)

        epoch_bar.set_postfix(avg_val=f"{avg_val_acc:.3f}", ksl=f"{val_results['ksl']:.3f}",
                               best_ksl=f"{best_ksl_acc:.3f}")

    print(f"  best avg val acc: {best_avg_acc:.4f}  |  ksl acc at that epoch: {best_ksl_acc:.4f}  ->  {ckpt_path}")
    return best_ksl_acc


## 7. Run all five configs

Resumable: any config already present in `RESULTS_PATH` is skipped, so a
crash or interrupt partway through only costs the configs not yet
finished. Set `FORCE_RERUN = True` to ignore saved results and redo
everything.

In [ ]:
FORCE_RERUN = False

results = {}
if RESULTS_PATH.exists() and not FORCE_RERUN:
    results = json.loads(RESULTS_PATH.read_text())
    print(f"resuming -- {len(results)} config(s) already done: {list(results)}")

epochs = EPOCHS_OVERRIDE or EPOCHS
for cfg in ABLATIONS:
    if cfg["name"] in results:
        print(f"skipping {cfg['name']} (already in {RESULTS_PATH})")
        continue
    t0 = time.time()
    acc = run_ablation(cfg, epochs)
    results[cfg["name"]] = {"label": cfg["label"], "accuracy": acc,
                             "seconds": time.time() - t0}
    RESULTS_PATH.write_text(json.dumps(results, indent=2))   # save after EVERY config

print("\nall done." if len(results) == len(ABLATIONS) else "\nstopped early -- rerun this cell to continue.")


## 8. Build Table 4

In [ ]:
full_acc = results["full_model"]["accuracy"] * 100

rows = []
for cfg in ABLATIONS:
    r = results.get(cfg["name"])
    if r is None:
        rows.append({"Configuration": cfg["label"], "Accuracy": None, "Δ vs. full model": None})
        continue
    acc_pct = r["accuracy"] * 100
    delta = None if cfg["name"] == "full_model" else round(acc_pct - full_acc, 2)
    rows.append({"Configuration": cfg["label"],
                 "Accuracy": round(acc_pct, 2),
                 "Δ vs. full model": delta})

table4 = pd.DataFrame(rows)
table4.to_excel("../plots/ablation_table4.xlsx", index=False)
table4


## 9. Paste-ready text for the paper

Prints each row as `Accuracy% | Δpp`, sorted by |Δ| descending (biggest
contributor first) -- matches the discussion-ordering convention used
elsewhere in the paper.

In [ ]:
ranked = sorted(
    (r for r in rows[1:] if r["Δ vs. full model"] is not None),
    key=lambda r: abs(r["Δ vs. full model"]), reverse=True,
)

print(f"{rows[0]['Configuration']}: {rows[0]['Accuracy']}%  (Δ --)")
for r in ranked:
    sign = "+" if r["Δ vs. full model"] > 0 else ""
    print(f"{r['Configuration']}: {r['Accuracy']}%  (Δ {sign}{r['Δ vs. full model']} pp)")
